# P3 Rutgers Catalog Data Quality Evidence

## tl;dr

- The frozen run produced 21 usable Catalog payloads covering all 15 currently exposed Fall 2026 campus values plus six Summer 2026 main/online scopes. Campus D was added through frozen amendment-002.
- The payloads contain 10,629 course rows, 22,069 section rows, and 30,804 meeting rows.
- Campus is mandatory in section identity: 3,125 `(term,index)` values collide across campuses. Eighteen raw `(term,campus,index)` duplicates are semantically equivalent after order-insensitive comment normalization; zero contradictory duplicates were observed.
- All 22 approved filter dimensions have a candidate source, but sparsity and reliability differ. Instructors expose names without stable IDs; 41 course strings have multiple objects, including 31 true supplement/credit/title variants.
- Open evidence contains 42/42 successful observations across two rounds. All values are five-digit strings; 14/21 raw target pairs changed and 3/21 Catalog-intersected states changed.
- The two-clock model is approved: public fixed 30 seconds, local default 30 seconds with range 3–3,600, and a 10-second related-batch target when actively watched.


## Context & Methods

This is a data-quality companion for the approved P2 contract. It reads only the Git-ignored raw responses named in the frozen request ledger and the deterministic profile emitted by `tools/profile_catalog_evidence.py`. It does not make network requests.

### Key Assumptions

- Counts describe the observed 2026 scopes, not an API SLA or all historical terms.
- Empty arrays for currently exposed off-campus values are treated as observed empty Catalogs, not source failures.
- Missing or sparse fields are not automatically defects; downstream three-valued matching must preserve uncertainty.
- Open values are interpreted only by official `(term,campus)` Catalog-set intersection; orphan values are audited and never create Sections.
- The notebook verifies frozen local evidence only and performs no Rutgers network requests.


In [1]:
import csv, hashlib, json
from pathlib import Path
p3 = Path.cwd() / 'project-governance' / 'current' / 'p3'
profile = json.loads((p3 / '04a-catalog-profile.json').read_text(encoding='utf-8'))
with (p3 / '02-catalog-request-ledger.tsv').open(encoding='utf-8', newline='') as f:
    ledger = list(csv.DictReader(f, delimiter='\t'))
success = [row for row in ledger if row['outcome'] == 'SUCCESS']
assert len(success) == profile['successPayloads'] == 21
for row in success:
    raw = Path(row['raw_relative_path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest().upper() == row['body_sha256']
print(f"ledger rows={len(ledger)}; usable payloads={len(success)}; raw hashes verified={len(success)}")


ledger rows=22; usable payloads=21; raw hashes verified=21


In [2]:
open_profile = json.loads((p3 / '13a-open-profile.json').read_text(encoding='utf-8'))
open_completion = json.loads((p3 / '22b-open-round2-completion.json').read_text(encoding='utf-8'))
open_decision = json.loads((p3 / '18a-open-review-decision-001.json').read_text(encoding='utf-8'))
open_contract = json.loads((p3 / '23b-shared-open-final-contract.json').read_text(encoding='utf-8'))
with (p3 / '11-open-request-ledger.tsv').open(encoding='utf-8', newline='') as f:
    open_ledger = list(csv.DictReader(f, delimiter='\t'))
assert len(open_ledger) == open_profile['successfulAttempts'] == open_completion['attempts']['successful'] == 42
assert sum(row['round'] == '1' for row in open_ledger) == 21
assert sum(row['round'] == '2' for row in open_ledger) == 21
assert all(row['outcome'] == 'SUCCESS' and row['http_status'] == '200' for row in open_ledger)
for row in open_ledger:
    raw = Path(row['raw_relative_path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest().upper() == row['body_sha256']
assert open_profile['status'] == 'ROUND_2_COMPLETE_OFFICIAL_JOIN_VALIDATED'
assert open_profile['valueTypeCounts']['string'] == open_profile['stringLengthCounts']['5'] == 302125
assert open_profile['totals']['officialJoinPassObservations'] == 42
assert open_profile['totals']['changedRawScopePairs'] == 14
assert open_profile['totals']['changedEffectiveScopePairs'] == 3
assert open_decision['proposedClockMapping']['status'] == 'APPROVED_BY_USER'
assert open_contract['status'] in {'P3_SHARED_OPEN_CONTRACT_FREEZE_CANDIDATE', 'FROZEN_P3_SHARED_OPEN_CONTRACT'}
print('Open: 42/42 hashes verified; 2 rounds; 302125 five-digit values')
print('Round-pair changes: raw=14/21; Catalog-intersected=3/21')
print('Clock mapping: public=30 fixed; local=30 range 3-3600; active-watch target=10')


Open: 42/42 hashes verified; 2 rounds; 302125 five-digit values
Round-pair changes: raw=14/21; Catalog-intersected=3/21
Clock mapping: public=30 fixed; local=30 range 3-3600; active-watch target=10


## Data

Each row below is one successfully cached term/campus Catalog payload. Four Fall off-campus scopes legitimately returned zero courses.

In [3]:
columns = ['term_id','campus','courses','sections','meetings','decoded_bytes']
print(' | '.join(columns))
print('-' * 88)
for row in profile['scopeSummary']:
    print(' | '.join(str(row[col]) for col in columns))


term_id | campus | courses | sections | meetings | decoded_bytes
----------------------------------------------------------------------------------------
92026 | NB | 4445 | 11933 | 17277 | 20920953
92026 | NK | 1333 | 2633 | 3835 | 4852179
92026 | CM | 969 | 1678 | 2209 | 3354814
92026 | ONLINE_NB | 838 | 1420 | 1496 | 2727357
92026 | ONLINE_NK | 199 | 225 | 229 | 490412
92026 | ONLINE_CM | 219 | 299 | 300 | 655131
92026 | B | 0 | 0 | 0 | 2
92026 | CC | 19 | 46 | 70 | 103159
92026 | H | 3 | 3 | 3 | 6155
92026 | CU | 0 | 0 | 0 | 2
92026 | MC | 0 | 0 | 0 | 2
92026 | L | 24 | 25 | 25 | 58633
92026 | AC | 18 | 22 | 24 | 60924
92026 | J | 0 | 0 | 0 | 2
72026 | NB | 1045 | 1697 | 2524 | 3553897
72026 | NK | 394 | 624 | 1013 | 1294383
72026 | CM | 213 | 271 | 344 | 625895
72026 | ONLINE_NB | 598 | 845 | 1026 | 1874930
72026 | ONLINE_NK | 146 | 165 | 218 | 380602
72026 | ONLINE_CM | 163 | 180 | 208 | 444061
92026 | D | 3 | 3 | 3 | 6911


## Results

### Composite section identity survives raw duplication only with explicit normalization

In [4]:
for item in profile['identity']['summary']:
    print(f"{item['metric']}: {item['count']} [{item['severity']}]")


section_rows: 22069 [INFO]
unique_term_campus_index: 22051 [INFO]
duplicate_term_campus_index_keys: 18 [RAW_DUPLICATE_REQUIRES_NORMALIZATION]
duplicate_term_campus_index_rows: 18 [RAW_DUPLICATE_REQUIRES_NORMALIZATION]
semantically_equivalent_duplicate_keys: 18 [SAFE_TO_COLLAPSE_WITH_AUDIT]
semantically_conflicting_duplicate_keys: 0 [CRITICAL_IF_NONZERO]
term_index_cross_campus_collisions: 3125 [EXPECTED_PROOF_COMPOSITE]
bare_index_cross_scope_collisions: 3344 [EXPECTED_PROOF_COMPOSITE]
multi_object_course_string_groups: 41 [COURSE_GROUP_NORMALIZATION_REQUIRED]
course_groups_with_multiple_variants: 31 [COURSE_VARIANT_MODEL_REQUIRED]
course_groups_with_section_overlap: 0 [CONFLICT_CHECK_REQUIRED]


### Delivery and meeting fields are rich but cannot be collapsed into one enum

In [5]:
print('Top raw meeting mode tuples:')
for row in profile['deliveryRawValues'][:15]:
    print(f"{row['meeting_mode_code']:>3} {row['meeting_mode_desc']:<34} meetings={row['meeting_count']} scopes={row['scope_count']}")
print('Meeting-day tokens:', {row['meeting_day']: row['meeting_count'] for row in profile['meetingDayValues']})
print('Time shapes:', profile['timeShapeCounts'])


Top raw meeting mode tuples:
 02 LEC                                meetings=13506 scopes=14
 90 ONLINE INSTRUCTION(INTERNET)       meetings=6442 scopes=16
 19 PROJ-IND                           meetings=2450 scopes=10
 23 RSCH-MA                            meetings=2238 scopes=5
 03 RECIT                              meetings=1264 scopes=7
 05 LAB                                meetings=1244 scopes=7
 08 MUS-INDV                           meetings=566 scopes=2
 04 SEM                                meetings=546 scopes=7
 07 STUDIO                             meetings=467 scopes=5
 80 GRADUATE 800-LEVEL                 meetings=406 scopes=5
 16 CLINIC                             meetings=303 scopes=7
 15 INTERNSP                           meetings=294 scopes=8
 09 MUS-GRP                            meetings=227 scopes=2
 91 HYBRID SECTION                     meetings=162 scopes=5
 92 REMOTE-SYNCH                       meetings=135 scopes=4
Meeting-day tokens: {'<EMPTY>': 12863, 'T': 42

### Every approved filter has a candidate source, with explicit sparsity

In [6]:
for row in profile['filterSourceProfile']:
    pct = 100 * row['non_empty_rate']
    print(f"{row['filter_id']} {row['label']:<28} {row['non_empty']}/{row['denominator']} ({pct:6.2f}%) — {row['evidence_note']}")


FLT-C01 Term                         21/21 (100.00%) — request scope
FLT-C02 Campus                       21/21 (100.00%) — official selector + request scope
FLT-C03 Subject                      10629/10629 (100.00%) — structured
FLT-C04 Text search                  10629/10629 (100.00%) — raw text exists; FTS ingest separately audited
FLT-C05 Course number                10629/10629 (100.00%) — structured
FLT-C06 Level                        10629/10629 (100.00%) — structured raw value
FLT-C07 Credits                      10629/10629 (100.00%) — structured object preferred
FLT-C08 Core code                    2686/10629 ( 25.27%) — legitimate sparse array
FLT-C09 Prerequisite presence        3056/10629 ( 28.75%) — empty vs unknown requires source rule
FLT-C10 Course campus location       10629/10629 (100.00%) — structured array
FLT-S01 Index                        22069/22069 (100.00%) — structured; composite scope required
FLT-S02 Section number               22069/22069 (100.00%) — 

### Two Open rounds validate target-local intersection while ETag remains audit-only

In [7]:
print('Open request duration (ms):', open_profile['requestDurationMs']['all'])
print('Headers:', open_profile['headerPresence'])
print('Transition totals:', {k: open_profile['totals'][k] for k in [
    'changedRawScopePairs', 'changedEffectiveScopePairs',
    'effectiveAddedSummedNonDistinct', 'effectiveRemovedSummedNonDistinct']})
print('Official join:', open_profile['officialJoinContract'])


Open request duration (ms): {'min': 1015.117, 'median': 1195.597, 'mean': 1241.159, 'p95NearestRank': 1501.02, 'max': 1537.168}
Headers: {'date': 42, 'etag': 42, 'age': 0, 'lastModified': 0}
Transition totals: {'changedRawScopePairs': 14, 'changedEffectiveScopePairs': 3, 'effectiveAddedSummedNonDistinct': 3, 'effectiveRemovedSummedNonDistinct': 8}
Official join: {'name': 'RUTGERS_OFFICIAL_MERGED_SET_INTERSECTION', 'scopeKey': 'term_campus', 'externalSectionKey': 'term_campus_index', 'orphanHandling': 'AUDIT_AND_IGNORE_NEVER_CREATE_SECTION', 'absenceToClosedRequires': 'COMPLETE_SAFE_NON_AMBIGUOUS_SCOPE_BATCH', 'individualEmptyWithNonemptyCatalog': 'UNSAFE_EMPTY_KEEP_LAST_KNOWN_GOOD', 'partialOrInvalidBatch': 'KEEP_LAST_KNOWN_GOOD'}


## Takeaways

1. Preserve `(term,campus,index)` and reject bare index or `(term,index)` joins. Collapse only semantically equivalent duplicate raw sections; quarantine contradictory duplicates.
2. Model a course-centered group separately from offering variants. A single course string may have supplement/credit/title variants or multiple disjoint section chunks.
3. Use official `sectionCourseType` for modality and retain every raw meeting-mode tuple. Synchronicity needs a conservative oracle; generic online remains unspecified unless independent source facts establish sync/async.
4. Treat empty Catalogs, sparse prerequisites/core/permission/eligibility, missing instructor IDs, and by-arrangement meetings as explicit data states. Do not replace them with fallback data or binary assumptions.
5. The 42 Open observations close the P3 evidence portion: the official target-local set intersection, two-clock mapping, empty/failure safety, scheduler, backoff, observations, counters, and episode semantics are consolidated in the final shared Open contract.
6. `Cache-Control: max-age=30` is observed response freshness metadata, not a Rutgers seat-publication SLA. The product may target <=1 second from an accepted valid observation to server fanout, but it must not promise a real seat change will alert within 30 seconds.
